In [ ]:
import xarray as xr
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd

## SelectStations

This notebook contains code that was used to subsample the dense SCHISM grid to keep only selected locations for training. Kept in the repository for reference, might need to be updated to match current or future training datasets.

In [ ]:
ds = xr.open_dataset("/work/ka1176/shared_data/2025-wwm-emulator-schism/processed_SNS_RUN00a_LevanteJP.zarr")

In [ ]:
out2d = xr.open_dataset("/work/bg1602/g260094/SNS/RUN00a/outputs_all/out2d_90.nc")

In [ ]:
ds

In [ ]:
out2d

In [ ]:
node_x = out2d.SCHISM_hgrid_node_x.values
node_y = out2d.SCHISM_hgrid_node_y.values

In [ ]:
plt.hist(node_x);

In [ ]:
plt.hist(node_y);

In [ ]:
node_y.min(), node_y.max()
xbins = np.arange(-180000, 590000, 10000)
ybins = np.arange(5550000, 6220000, 10000)
len(ybins) * len(xbins)

In [ ]:
hist, xb, yb, _ = plt.hist2d(node_x, node_y, bins=(xbins, ybins), norm="log");
plt.colorbar()

In [ ]:
# todo select one node per bin and store the index

In [ ]:
from __future__ import annotations

from typing import Iterable, Optional, Tuple
import numpy as np


def select_one_point_per_2d_bin(
    x: Iterable[float],
    y: Iterable[float],
    x_bins: Iterable[float],
    y_bins: Iterable[float],
    *,
    keep: str = "first",          # "first" | "random" | "closest_center"
    seed: Optional[int] = None,   # used when keep="random"
    return_indices: bool = True,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray] | Tuple[np.ndarray, np.ndarray]:
    """
    Select at most one point per 2D bin defined by x_bins and y_bins.

    Parameters
    ----------
    x, y
        1D arrays/lists of coordinates with equal length N.
    x_bins, y_bins
        Bin edges (like numpy.histogram). Must be monotonically increasing.
        A point is inside if x_bins[i] <= x < x_bins[i+1] (same for y),
        except the last edge is excluded like np.digitize(..., right=False).
    keep
        How to choose a representative point if multiple fall in the same bin:
        - "first": first occurrence
        - "random": random among points in the bin
        - "closest_center": point closest to the bin center (euclidean in x/y)
    seed
        RNG seed for keep="random".
    return_indices
        If True, also return indices into the original arrays.

    Returns
    -------
    (x_sel, y_sel, idx_sel) if return_indices else (x_sel, y_sel)
    """
    x = np.asarray(x)
    y = np.asarray(y)
    if x.shape != y.shape:
        raise ValueError("x and y must have the same shape")
    if x.ndim != 1:
        raise ValueError("x and y must be 1D")

    x_bins = np.asarray(x_bins)
    y_bins = np.asarray(y_bins)
    if x_bins.ndim != 1 or y_bins.ndim != 1:
        raise ValueError("x_bins and y_bins must be 1D bin-edge arrays")
    if len(x_bins) < 2 or len(y_bins) < 2:
        raise ValueError("x_bins and y_bins must have at least 2 edges")

    # Bin index per point (0..nbins-1); points outside get -1
    xb = np.digitize(x, x_bins) - 1
    yb = np.digitize(y, y_bins) - 1

    nx = len(x_bins) - 1
    ny = len(y_bins) - 1
    inside = (xb >= 0) & (xb < nx) & (yb >= 0) & (yb < ny)

    idx = np.nonzero(inside)[0]
    if idx.size == 0:
        empty = np.array([], dtype=int)
        if return_indices:
            return x[:0], y[:0], empty
        return x[:0], y[:0]

    xb = xb[inside]
    yb = yb[inside]
    x_in = x[inside]
    y_in = y[inside]

    # Unique 2D bin key
    key = xb.astype(np.int64) * ny + yb.astype(np.int64)

    if keep == "first":
        # Sort by key, keep first per key, then return in original order
        order = np.argsort(key, kind="stable")
        key_sorted = key[order]
        first_mask = np.ones_like(key_sorted, dtype=bool)
        first_mask[1:] = key_sorted[1:] != key_sorted[:-1]
        chosen_local = order[first_mask]

    elif keep == "random":
        rng = np.random.default_rng(seed)
        # Group by key in sorted order; choose one random index per group
        order = np.argsort(key, kind="stable")
        key_sorted = key[order]
        starts = np.r_[0, np.nonzero(key_sorted[1:] != key_sorted[:-1])[0] + 1]
        ends = np.r_[starts[1:], key_sorted.size]
        chosen_local = np.array(
            [order[s + rng.integers(0, e - s)] for s, e in zip(starts, ends)],
            dtype=int,
        )

    elif keep == "closest_center":
        # Compute squared distance to bin center for each point, then take argmin per key
        x_center = (x_bins[xb] + x_bins[xb + 1]) / 2.0
        y_center = (y_bins[yb] + y_bins[yb + 1]) / 2.0
        d2 = (x_in - x_center) ** 2 + (y_in - y_center) ** 2

        order = np.argsort(key, kind="stable")
        key_sorted = key[order]
        d2_sorted = d2[order]

        starts = np.r_[0, np.nonzero(key_sorted[1:] != key_sorted[:-1])[0] + 1]
        ends = np.r_[starts[1:], key_sorted.size]

        chosen_local = []
        for s, e in zip(starts, ends):
            best = s + int(np.argmin(d2_sorted[s:e]))
            chosen_local.append(order[best])
        chosen_local = np.asarray(chosen_local, dtype=int)

    else:
        raise ValueError('keep must be "first", "random", or "closest_center"')

    # Map chosen_local (indices in inside-filtered arrays) back to original indices
    idx_sel = idx[chosen_local]

    # Return in increasing original index order (often convenient)
    idx_sel = np.sort(idx_sel)
    x_sel = x[idx_sel]
    y_sel = y[idx_sel]

    if return_indices:
        return x_sel, y_sel, idx_sel
    return x_sel, y_sel

In [ ]:
xs, ys, ii = select_one_point_per_2d_bin(node_x, node_y, xbins, ybins, keep="closest_center")
print(xs, ys, ii)

In [ ]:
ii

In [ ]:
plt.scatter(node_x[::10], node_y[::10], marker="x", alpha=0.1)
#plt.scatter(node_x[231704], node_y[231704], color="red")
#plt.scatter(xy.iloc[:, 0], xy.iloc[:, 1])
plt.scatter(out2d.isel(nSCHISM_hgrid_node=ii).SCHISM_hgrid_node_x.values, out2d.isel(nSCHISM_hgrid_node=ii).SCHISM_hgrid_node_y.values, marker="x")

In [ ]:
ds.isel(nSCHISM_hgrid_node=ii)["sigWaveHeight"].mean(dim="nSCHISM_hgrid_node").plot()

In [ ]:
selection = pd.DataFrame({"nSCHISM_hgrid_node": ii})
selection.to_csv("../src/data/node_idx_closest_center_SNS00a_10000.csv", index=False)

In [ ]:
# TODO: Boundary nodes in this grid